
---

## 🧠 Interview Transcript Evaluation

### Language, Authenticity, and AI-Script Detection (Process Digital Twin – Step 4)

This notebook implements the **language-level analysis stage** of the agent-based digital twin described in
*“An Agent-Based Digital Twin for Transparent AI in Student Admissions.”*

In high-stakes admission workflows, interview answers are not evaluated only for *what* is said, but also for *how* it is said.
Assessing communication quality, reasoning depth, and authenticity requires agents that can operate on **ambiguous, subjective, and linguistically complex material**, where simple rules or keyword checks are insufficient.

This notebook demonstrates how such reasoning can be externalized into an **inspectable AI agent**, operating on interview transcripts generated in earlier steps.

---

### What this agent evaluates

For each interview answer, the agent analyzes:

* **Relevance and content quality**
  Does the response address the question with concrete, personal, and coherent information?

* **Language structure and clarity**
  Is the answer well structured and understandable, or vague and generic?

* **Natural speech vs. scripted delivery**
  Does the transcript resemble spontaneous spoken language, or a read-out essay?

* **Markers of AI-generated text**
  Detection of linguistic patterns often associated with LLM-produced scripts (overly polished phrasing, characteristic vocabulary, unnatural sentence balance).

Each transcript is processed **independently**, reflecting how multiple AI agents would normally operate **in parallel** within a production admission system.

---

### Role in the digital twin

In the full system described in the paper, this stage can be executed by:

* local agents,
* cloud-based API agents operating on **anonymized transcripts**, or
* ensembles of specialized evaluators.

This notebook intentionally uses a **single, prompt-driven agent** to serve as a **reproducible proof-of-concept**.
Its purpose is not to claim optimal performance, but to show that **end-to-end language evaluation can be replayed, inspected, and audited** as part of the process digital twin.

Flattening this step into a standalone notebook allows readers to:

* inspect the exact evaluation logic,
* reproduce results,
* and understand how subjective judgments are operationalized by agents.

---

### Inputs

* Interview transcripts generated in Step 3
  (`video_interviews_audio_transcriptions_text/`)

### Outputs

* Structured, machine-readable evaluation results (JSON) per interview answer
  (`video_interviews_transcriptions_grade_info_*`)

---

### Prompt-based evaluation and transparency

All evaluations are driven by a **fixed expert prompt** simulating the role of an interview evaluator and forensic linguist.
This design choice supports:

* reproducibility across runs,
* explicit evaluation criteria,
* and traceable reasoning artifacts suitable for human review.

---

### Pipeline context

This notebook follows:

* **Step 1** – Interview recording (video capture)
* **Step 2** – Audio extraction
* **Step 3** – Speech-to-text transcription

It completes the **language and authenticity analysis stage** of the interview pipeline, before multimodal fusion and process-level reasoning in later steps.

---


In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

✅ Google Drive is already mounted


In [ ]:
import os

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"
os.makedirs(drive_dir, exist_ok=True)

subdirs = [d for d in os.listdir(drive_dir) if os.path.isdir(os.path.join(drive_dir, d))]
print(subdirs)

['0_0', '18190_27580', '40529_33700', '60811_46042']


In [ ]:
from pathlib import Path

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"

# Dictionary to store files by subdirectory
files_by_subdir = {}

# Iterate through direct subdirectories
for subdir in Path(drive_dir).iterdir():
    if subdir.is_dir():
        # Check if 'video_interviews' exists in this subdirectory
        documents_video_path = subdir / "video_interviews_audio_transcriptions_text"

        if documents_video_path.exists() and documents_video_path.is_dir():
            # Store both direct files and files in subdirectories
            video_info = {
                'direct_files': [],
                'subdirs': {}
            }

            # Iterate through contents of video_interviews
            for item in documents_video_path.iterdir():
                if item.is_file():
                    # Direct video file
                    video_info['direct_files'].append(item)
                elif item.is_dir():
                    # Subdirectory with video files
                    subdir_files = [f for f in item.iterdir() if f.is_file()]
                    if subdir_files:
                        video_info['subdirs'][item.name] = subdir_files

            files_by_subdir[subdir.name] = video_info

            total_files = len(video_info['direct_files']) + sum(len(files) for files in video_info['subdirs'].values())
            print(f"Found {total_files} files in {subdir.name}/video_interviews/")

# Now you can use the files later
for subdir_name, video_info in files_by_subdir.items():
    print(f"\n{subdir_name}:")

    # Print direct files
    if video_info['direct_files']:
        print("  Direct files:")
        for file in video_info['direct_files']:
            print(f"     {file.name}")

    # Print files in subdirectories
    if video_info['subdirs']:
        print("  Files in subdirectories:")
        for subdir_name, files in video_info['subdirs'].items():
            print(f"    {subdir_name}/")
            for file in files:
                print(f"       {file.name}")

Found 7 files in 60811_46042/video_interviews/

60811_46042:
  Direct files:
     51375_What_new_hobbies_or_activities_would_you_like_to_t.txt
     51376_Which_qualities_of_VILNIUS_TECH_stood_out_to_you_a.txt
     51377_How_would_your_friends_characterize_you_.txt
     51378_How_do_you_deal_with_unexpected_changes_or_disrupt.txt
     51379_What_does_innovation_mean_to_you_personally__how_d.txt
     51380_What_methods_do_you_use_to_make_sure_you_stay_on_t.txt
     51381_What_made_you_choose_this_specific_programme_at_VI.txt


In [ ]:
!pip install -U -q "google"
!pip install -U -q "google.genai"

import os
from google.colab import userdata
from google.colab import drive
os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Please ensure that uploaded files are available in the AI Studio folder or change the working folder.
#os.chdir("/content/drive/MyDrive/Google AI Studio")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [ ]:
import requests

prompt = requests.get("https://raw.githubusercontent.com/fair-vid/admission_prompts/main/video_analysis/interview_transcript_auditor_prompt.txt").text
#prompt = response.text

print(prompt[:2222])


# Role
You are an expert Interview Evaluator and Forensic Linguist. You specialize in analyzing video interview transcripts to assess communication skills and **detect the use of AI-generated scripts.**

# Task
Evaluate the provided interview transcript based on the one question asked. 
You must critically assess if the student is speaking naturally or reading a pre-written response generated by an LLM (like ChatGPT).

# Input Data
**Question asked:**
[Question]

**Student Transcript:**
[Interview Transcript]

# Evaluation Criteria
Please analyze the response based on the following metrics:
1. **Relevance & Content:** Did they answer the question with specific, personal examples?
2. **Naturalness vs. Scripted:** Does the speech flow like a natural conversation, or does it sound like a read essay?
3. **AI Markers:** Look for "hallmark" LLM words (e.g., "delve," "underscore," "pivotal role," "tapestry") and overly perfect grammar lacking natural speech pauses.

# Output Format
You must o

In [ ]:
import os
from google import genai
from google.genai import types

def generate_description(filled_prompt, model):

    client = genai.Client()

    # Prepare content
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=filled_prompt),
            ],
        ),
    ]

    generate_content_config = types.GenerateContentConfig()

    description = ""
    try:
        for chunk in client.models.generate_content_stream(
            model=model,
            contents=contents,
            config=generate_content_config,
        ):
            if chunk.text:
                description += chunk.text
    except Exception as e:
        print(f"An error occurred while processing: {e}")
        return None

    return description


# Example usage:
# result = generate_description(
#     filled_prompt="Your prompt here",
#     output_dir="/content/image-descriptions/",
#     filename="description.txt",
#     model="gemma-3-27b-it"
# )

In [ ]:
#@title 🛠️ Available models:
#from google import genai

client = genai.Client()

print("")
models = list(client.models.list())
for i, model in enumerate(sorted(models, key=lambda m: m.name.split('/')[-1])):
    model_name = model.name.split('/')[-1]
    print(f"{model_name:<50}", end="" if (i + 1) % 3 != 0 else "\n")




antigravity-preview-05-2026                       aqa                                               deep-research-max-preview-04-2026                 
deep-research-preview-04-2026                     deep-research-pro-preview-12-2025                 gemini-2.5-computer-use-preview-10-2025           
gemini-2.5-flash                                  gemini-2.5-flash-image                            gemini-2.5-flash-lite                             
gemini-2.5-flash-native-audio-latest              gemini-2.5-flash-native-audio-preview-09-2025     gemini-2.5-flash-native-audio-preview-12-2025     
gemini-2.5-flash-preview-tts                      gemini-2.5-pro                                    gemini-2.5-pro-preview-tts                        
gemini-3-flash-preview                            gemini-3-pro-image                                gemini-3-pro-image-preview                        
gemini-3.1-flash-image                            gemini-3.1-flash-image-preview             

In [ ]:
def clean_filename(filename):
    """Remove number prefix and convert underscores to spaces, remove extension"""
    import re
    # Remove number prefix (e.g., "1_")
    cleaned = re.sub(r'^\d+_', '', filename)
    # Remove extension
    cleaned = cleaned.rsplit('.', 1)[0]
    # Replace underscores with spaces
    cleaned = cleaned.replace('_', ' ')
    return cleaned

In [ ]:
import os
from pathlib import Path
import time # Import the time module

model = "gemma-4-31b-it"  # gemma-4-31b-it

# Process each subdirectory
for subdir_name, video_info in files_by_subdir.items():
    print(f"\nProcessing grading for {subdir_name}...")


    # Create grade info directory path
    grade_base_dir = Path(drive_dir) / subdir_name / ("video_interviews_transcriptions_grade_info"+"_"+model)
    transcription_base_dir = Path(drive_dir) / subdir_name / "video_interviews_audio_transcriptions_text"

    has_grade_files = False

    # Process direct transcript files
    if video_info['direct_files']:
        for video_file in video_info['direct_files']:
            # Extract question from filename
            question = clean_filename(video_file.name)

            # Prepare paths
            transcript_filename = video_file.stem + ".txt"
            transcript_path = transcription_base_dir / transcript_filename

            grade_filename = video_file.stem + ".txt"
            grade_path = grade_base_dir / grade_filename

            # Check if grade already exists
            if grade_path.exists():
                print(f"  ⏭️  Skipping {transcript_filename} (grade already exists)")
                has_grade_files = True
                continue

            # Check if transcript file exists
            if not transcript_path.exists():
                print(f"  ⚠️  Transcript file not found: {transcript_filename}")
                continue

            # Read transcript
            try:
                with open(transcript_path, "r", encoding="utf-8") as f:
                    transcript = f.read()
            except Exception as e:
                print(f"  ❌ Error reading {transcript_filename}: {e}")
                continue

            # Create grade directory if needed
            if not has_grade_files:
                os.makedirs(grade_base_dir, exist_ok=True)
                has_grade_files = True

            # Fill prompt
            filled_prompt = prompt.replace('[Question]', question).replace('[Interview Transcript]', transcript)

            # Generate description/grade
            try:
                print(f"  🤖 Grading {transcript_filename}")
                print(f"     Question: {question}")

                description = generate_description(  filled_prompt=filled_prompt,   model=model                )
                with open(grade_path, 'w') as f:
                    f.write(description)

                if description:
                    print(f"     ✓ Grade saved")
                else:
                    print(f"     ⚠️  No grade generated")

            except Exception as e:
                print(f"  ❌ Error grading {transcript_filename}: {e}")

    # Process files in subdirectories
    if video_info['subdirs']:
        for subdir_folder_name, files in video_info['subdirs'].items():
            # Create corresponding subdirectory in grade folder
            grade_subdir = grade_base_dir / subdir_folder_name
            transcription_subdir = transcription_base_dir / subdir_folder_name

            for video_file in files:
                # Extract question from filename
                question = clean_filename(video_file.name)

                # Prepare paths
                transcript_filename = video_file.stem + ".txt"
                transcript_path = transcription_subdir / transcript_filename

                grade_filename = video_file.stem + ".txt"
                grade_path = grade_subdir / grade_filename

                # Check if grade already exists
                if grade_path.exists():
                    print(f"  ⏭️  Skipping {subdir_folder_name}/{transcript_filename} (grade already exists)")
                    has_grade_files = True
                    continue

                # Check if transcript file exists
                if not transcript_path.exists():
                    print(f"  ⚠️  Transcript file not found: {subdir_folder_name}/{transcript_filename}")
                    continue

                # Read transcript
                try:
                    with open(transcript_path, "r", encoding="utf-8") as f:
                        transcript = f.read()
                except Exception as e:
                    print(f"  ❌ Error reading {subdir_folder_name}/{transcript_filename}: {e}")
                    continue

                # Create grade subdirectory if needed
                if not has_grade_files:
                    os.makedirs(grade_base_dir, exist_ok=True)
                    has_grade_files = True
                os.makedirs(grade_subdir, exist_ok=True)

                # Fill prompt
                filled_prompt = prompt.replace('[Question]', question).replace('[Interview Transcript]', transcript)

                # Generate description/grade
                try:
                    print(f"  🤖 Grading {subdir_folder_name}/{transcript_filename}")
                    print(f"     Question: {question}")

                    description = generate_description(  filled_prompt=filled_prompt,   model=model                )
                    with open(grade_path, 'w') as f:
                        f.write(description)


                    if description:
                        print(f"     ✓ Grade saved")
                    else:
                        print(f"     ⚠️  No grade generated")

                except Exception as e:
                    print(f"  ❌ Error grading {subdir_folder_name}/{transcript_filename}: {e}")

    time.sleep(5)

print("\n✅ All grading completed!")


Processing grading for 60811_46042...


  🤖 Grading 51375_What_new_hobbies_or_activities_would_you_like_to_t.txt
     Question: What new hobbies or activities would you like to t
     ✓ Grade saved
  🤖 Grading 51376_Which_qualities_of_VILNIUS_TECH_stood_out_to_you_a.txt
     Question: Which qualities of VILNIUS TECH stood out to you a
     ✓ Grade saved
  🤖 Grading 51377_How_would_your_friends_characterize_you_.txt
     Question: How would your friends characterize you 
     ✓ Grade saved
  🤖 Grading 51378_How_do_you_deal_with_unexpected_changes_or_disrupt.txt
     Question: How do you deal with unexpected changes or disrupt
     ✓ Grade saved
  🤖 Grading 51379_What_does_innovation_mean_to_you_personally__how_d.txt
     Question: What does innovation mean to you personally  how d
     ✓ Grade saved
  🤖 Grading 51380_What_methods_do_you_use_to_make_sure_you_stay_on_t.txt
     Question: What methods do you use to make sure you stay on t
     ✓ Grade saved
  🤖 Grading 51381_What_made_you_choose_this_specific_programme_at_VI.txt


---

## 🔗 Next Step: Visual Behavior & Non-Verbal Evidence

Language-based evaluation alone cannot fully explain how an interview answer was produced.
In the proposed digital twin architecture, **textual reasoning is complemented by independent visual evidence** derived directly from interview video.

The next notebook introduces a **vision-based perception agent** that analyzes non-verbal interview behavior, including:

* head motion stability,
* gaze aversion and eye-scanning patterns,
* blink rate and facial expressivity,
* indicators of reading or scripted delivery while speaking.

These signals provide **orthogonal evidence** to transcript-based analysis and are designed to be fused later with linguistic and acoustic outputs at the process level.

### ▶ Open the Visual Behavior Analysis notebook:

**Multimodal Interview Behavior Analysis (MediaPipe Agent)**
[https://colab.research.google.com/drive/1s4_BvYwYsaTTuakjWip8A79VdepowvgY?usp=sharing](https://colab.research.google.com/drive/1s4_BvYwYsaTTuakjWip8A79VdepowvgY?usp=sharing)

Together, the transcript evaluation and visual behavior agents illustrate how the digital twin decomposes subjective assessment into **transparent, auditable, and reproducible AI components**, each operating on a distinct modality while contributing to a unified decision model.

---
